# 04 · Schrödinger ground state (QPINN)

`omnibias-qpinn` builds quantum PINNs on top of the closed-form derivative
kernels. Here we find the ground state of the 1D **quantum harmonic
oscillator**,

$$-\tfrac12 \psi'' + \tfrac12 x^2 \psi = E\,\psi,$$

whose exact ground state is $\psi_0(x)=\pi^{-1/4}e^{-x^2/2}$ with energy
$E_0=\tfrac12$ (atomic units). The kinetic term $\psi''$ is closed-form, so the
TISE residual is bit-stable. Runs on CPU in well under a minute.

In [ ]:
import sys
import torch
import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, ".")
from _style import set_style, ACCENT, GOOD, PRIMARY, INK
set_style()

from omnibias.pinn._core.coords import CoordinateSpec
from omnibias.pinn.torch.fields.one_layer import OneLayerVectorField
from omnibias.qpinn import make_psi_components, psi_density
from omnibias.qpinn.torch.cage import norm_loss
from omnibias.qpinn.torch.equations import tise

torch.manual_seed(0)

coord = CoordinateSpec(("x",))
field = OneLayerVectorField(
    coordinate_spec=coord, components=make_psi_components(name="psi"),
    hidden=32, base="gaussian", dtype=torch.float64,
)
xs = torch.linspace(-4.0, 4.0, 401, dtype=torch.float64).unsqueeze(-1)
ws = torch.full((401,), 8.0 / 401, dtype=torch.float64)

def potential(state):
    return 0.5 * state.coords[..., 0] ** 2

## Train to the ground state

We minimise the TISE residual at the known energy `E = 0.5` plus a soft
normalisation penalty, and track the energy estimate.

In [ ]:
optim = torch.optim.Adam(field.parameters(), lr=5e-3)
energy_hist = []
for step in range(1500):
    optim.zero_grad()
    state = field(xs)
    out = tise(state, energy=0.5, potential=potential, quadrature_weights=ws)
    loss = (out.residual**2).sum(dim=-1).mean() + 10.0 * norm_loss(
        state, quadrature_weights=ws, target_norm=1.0)
    loss.backward(); optim.step()
    energy_hist.append(float(out.energy_estimate.detach()))
    if step % 300 == 0:
        print(f"step {step:5d}  loss={float(loss):.3e}  E_est={energy_hist[-1]:.6f}")
print(f"final E_est = {energy_hist[-1]:.6f}   (exact E_0 = 0.5)")

In [ ]:
with torch.no_grad():
    dens = psi_density(field(xs)).squeeze(-1).numpy()
x = xs.squeeze(-1).numpy()
dens = dens / np.trapz(dens, x)              # normalise for the comparison
analytic = np.exp(-x**2) / np.sqrt(np.pi)    # |psi_0|^2

fig, (axl, axr) = plt.subplots(1, 2, figsize=(11, 4.2))
axl.plot(energy_hist, color=PRIMARY)
axl.axhline(0.5, color=ACCENT, ls="--", label="exact $E_0=1/2$")
axl.set_xlabel("step"); axl.set_ylabel("energy estimate"); axl.set_title("Energy convergence"); axl.legend()

axr.plot(x, analytic, color=ACCENT, lw=3, alpha=0.6, label="exact $|\\psi_0|^2$")
axr.plot(x, dens, color=INK, ls="--", label="QPINN")
axr.set_xlabel("x"); axr.set_title("Ground-state density"); axr.legend()
plt.tight_layout(); plt.show()
print(f"energy error = {abs(energy_hist[-1] - 0.5):.2e}")

## Takeaway

A tiny one-layer field recovers the harmonic-oscillator ground state and its
energy `E₀ = ½` from a closed-form-kinetic TISE residual. `omnibias-qpinn` also
ships TDSE, Gross–Pitaevskii (solitons), Helmholtz, Klein–Gordon, and Dirac
residuals with the same machinery.

Next: **[05 · local kinetic energy for VMC/FermiNet](05_local_kinetic_energy.ipynb)**.